<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/10_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ======================================================================
# 1. 드라이브 연결, 원시자료 로드 및 전처리 (df 정의)
# ======================================================================
from google.colab import drive
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 드라이브 마운트
drive.mount('/content/drive')

# 청소년건강행태조사 데이터 불러오기
DATA_PATH = '/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat'
df = pd.read_sas(DATA_PATH)

# 고등학생 도메인 지정 (GRADE 4, 5, 6)
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)

# 아침식사 결식 변수 (F_BR 1~3: 주 5회 이상 결식 = 결식군 1)
conditions_bf = [df['F_BR'].isin([1, 2, 3]), df['F_BR'].isin([4, 5, 6, 7, 8])]
df['bf_skip'] = np.select(conditions_bf, [1, 0], default=np.nan)

# BMI 계산 및 비만 여부 (BMI 25 이상)
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)
df['obesity'] = np.where(df['bmi'] >= 25, 1, np.where(df['bmi'].notnull(), 0, np.nan))

# 유효 대상자 설정 (25,077명)
df['valid_sample'] = (df['domain_hs'] == 1) & df['bf_skip'].notna() & df['bmi'].notna()
valid_df = df[df['valid_sample'] == True].copy()


# ======================================================================
# 2. B-2 Table 1 전체 수치 및 복합표본 SE 자동 산출
# ======================================================================
def calc_complex_mean_se(data, var_name):
    model = sm.GLM(data[var_name], np.ones(len(data)), family=sm.families.Gaussian(), freq_weights=data['W']).fit(
        cov_type='cluster', cov_kwds={'groups': data['CLUSTER']}
    )
    return model.params[0], model.bse[0]

def calc_cat_table(data, var_name):
    unweighted_n = data[var_name].value_counts().sort_index()
    weighted_sum = data.groupby(var_name)['W'].sum().sort_index()
    weighted_pct = (weighted_sum / data['W'].sum()) * 100
    return pd.DataFrame({'n': unweighted_n, 'pct': weighted_pct})

df_total = valid_df
df_skip = valid_df[valid_df['bf_skip'] == 1]
df_noskip = valid_df[valid_df['bf_skip'] == 0]

print(f"=== Table 1 분석 대상 인원 ===")
print(f"전체: n={len(df_total):,}, 가중% = 100.0%")
print(f"결식군: n={len(df_skip):,}, 가중% = {(df_skip['W'].sum()/df_total['W'].sum())*100:.2f}%")
print(f"비결식군: n={len(df_noskip):,}, 가중% = {(df_noskip['W'].sum()/df_total['W'].sum())*100:.2f}%\n")

for var, name in [("SEX", "성별 (1: 남, 2: 여)"), ("GRADE", "학년 (4: 고1, 5: 고2, 6: 고3)"), ("obesity", "비만 여부 (0: 비비만, 1: 비만)")]:
    print(f"--- [{name}] (n, 가중 %) ---")
    t_tot = calc_cat_table(df_total, var)
    t_sk = calc_cat_table(df_skip, var)
    t_nsk = calc_cat_table(df_noskip, var)
    res = pd.DataFrame({
        '전체 n': t_tot['n'], '전체 가중%': t_tot['pct'].round(2),
        '결식군 n': t_sk['n'], '결식군 가중%': t_sk['pct'].round(2),
        '비결식군 n': t_nsk['n'], '비결식군 가중%': t_nsk['pct'].round(2)
    })
    print(res)
    print()

print("--- [BMI (kg/m²)] 가중 평균 (SE) ---")
m_tot, se_tot = calc_complex_mean_se(df_total, 'bmi')
m_sk, se_sk = calc_complex_mean_se(df_skip, 'bmi')
m_nsk, se_nsk = calc_complex_mean_se(df_noskip, 'bmi')

print(f"전체 BMI: {m_tot:.2f} (SE: {se_tot:.4f})")
print(f"결식군 BMI: {m_sk:.2f} (SE: {se_sk:.4f})")
print(f"비결식군 BMI: {m_nsk:.2f} (SE: {se_nsk:.4f})")

Mounted at /content/drive


/tmp/ipykernel_903/3303926387.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
/tmp/ipykernel_903/3303926387.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['bf_skip'] = np.select(conditions_bf, [1, 0], default=np.nan)
/tmp/ipykernel_903/3303926387.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) 

=== Table 1 분석 대상 인원 ===
전체: n=25,077, 가중% = 100.0%
결식군: n=11,376, 가중% = 45.68%
비결식군: n=13,701, 가중% = 54.32%

--- [성별 (1: 남, 2: 여)] (n, 가중 %) ---
      전체 n  전체 가중%  결식군 n  결식군 가중%  비결식군 n  비결식군 가중%
SEX                                                 
1.0  12949   51.55   5697    50.02    7252     52.84
2.0  12128   48.45   5679    49.98    6449     47.16

--- [학년 (4: 고1, 5: 고2, 6: 고3)] (n, 가중 %) ---
       전체 n  전체 가중%  결식군 n  결식군 가중%  비결식군 n  비결식군 가중%
GRADE                                                
4.0    8526   32.83   3676    31.23    4850     34.18
5.0    8363   33.17   3903    34.13    4460     32.36
6.0    8188   34.00   3797    34.64    4391     33.46

--- [비만 여부 (0: 비비만, 1: 비만)] (n, 가중 %) ---
          전체 n  전체 가중%  결식군 n  결식군 가중%  비결식군 n  비결식군 가중%
obesity                                                 
0.0      19933   79.94   8976    79.35   10957     80.44
1.0       5144   20.06   2400    20.65    2744     19.56

--- [BMI (kg/m²)] 가중 평균 (SE) ---
전체 BMI: 22.18 (SE: 0.

/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(
/tmp/ipykernel_903/3303926387.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return model.params[0], model.bse[0]
/tmp/ipykernel_903/3303926387.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return model.params[0], model.bse[0]
/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(
/tmp